In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, TrainingArguments, Trainer
import torch

model_checkpoint = "nguyenvulebinh/vi-mrc-large"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

# Load your dataset
data_files = {
    "train": "/home/annv/git/extractive-qa-mrc/data-bin/my-data/train.json",
    "validation": "/home/annv/git/extractive-qa-mrc/data-bin/my-data/validation.json"}
dataset = load_dataset("json", data_files=data_files)
dataset = dataset.with_format("torch")

# Preprocessing
def prepare_features(example):
    tokenized = tokenizer(
        example["question"],
        example["context"],
        truncation="only_second",
        max_length=512,
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )
    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        sequence_ids = tokenized.sequence_ids(i)

        sample_index = sample_mapping[i]
        answers = example["answers"][sample_index]
        if len(answers["answer_start"]) == 0:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])

        token_start_index = 0
        while sequence_ids[token_start_index] != 1:
            token_start_index += 1

        token_end_index = len(input_ids) - 1
        while sequence_ids[token_end_index] != 1:
            token_end_index -= 1

        if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
        else:
            while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                token_start_index += 1
            start_positions.append(token_start_index - 1)

            while token_end_index >= 0 and offsets[token_end_index][1] >= end_char:
                token_end_index -= 1
            end_positions.append(token_end_index + 1)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    return tokenized

tokenized_datasets = dataset.map(prepare_features, batched=True, remove_columns=dataset["train"].column_names)

# Training arguments
training_args = TrainingArguments(
    output_dir="./vi-mrc-finetuned",
    # eval_strategy="steps",
    eval_steps=50,
    save_steps=100,
    logging_steps=10,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=20,
    save_total_limit=1,
    learning_rate=3e-5,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets.get("validation"),
    tokenizer=tokenizer,
)

trainer.train()


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

/tmp/ipykernel_32905/608163329.py:91: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/home/annv/.local/lib/python3.10/site-packages/transformers/utils/generic.py:271: FutureWarning: The input object of type 'Tensor' is an array-like implementing one of the corresponding protocols (`__array__`, `__array_interface__` or `__array_struct__`); but not a sequence (or 0-D). In the future, this object will be coerced as if it was first converted using `np.array(obj)`. To retain the old behaviour, you have to either modify the type 'Tensor', or assign to an empty array created with `np.empty(correct_shape, dtype=object)`.
  arr = np.array(obj)


Step,Training Loss
10,1.822100
20,0.117500
30,0.000200
40,0.042400


TrainOutput(global_step=40, training_loss=0.49554611031198875, metrics={'train_runtime': 364.4624, 'train_samples_per_second': 0.274, 'train_steps_per_second': 0.11, 'total_flos': 92870699212800.0, 'train_loss': 0.49554611031198875, 'epoch': 20.0})

In [ ]:
# !pip install torch --upgrade
# !pip install transformers datasets accelerate evaluate


In [ ]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

# Load tokenizer and model from your finetuned directory
model_path = "./vi-mrc-finetuned/checkpoint-40"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForQuestionAnswering.from_pretrained(model_path)

# Put model in evaluation mode
model.eval()

# Your test input
# question = "tam nguyên phạm đôn lễ sinh năm nào" # Answer: năm 14
# context = "Phạm Đôn Lễ sinh năm 1457 tại làng Hải Triều (tục gọi là làng Hới), thuộc tổng Thanh Triều, phủ Long Hưng, huyện Ngự Thiên, tỉnh Hưng Yên (nay là thôn Hải Triều, xã Tân Lễ, huyện Hưng Hà, tỉnh Thái Bình) trong một gia đình nghèo khó, bố làm nghề chài lưới, mẹ bán quán nước cho khách qua đò. Khi ông còn rất nhỏ thì người bố qua đời, hai mẹ con đơn côi sống dựa vào hàng quán. Một lần Phạm Đôn Lễ bị lạc ở bờ sông Luộc, người mẹ đi tìm khắp nơi mà không được. Trong lúc lang thang vì lạc mẹ thì Phạm Đôn Lễ được một gia đình giàu có quê ở Thanh Hóa đưa lên thuyền về nhà nuôi dưỡng."
# context = "Tam Nguyên Phạm Đôn Lễ sinh năm 1457, mất năm 1531, là vị Tam Nguyên đầu tiên trong lịch sử khoa bảng Việt Nam. Ông đỗ Trạng nguyên năm 1481 dưới triều vua Lê Thánh Tông. Ông làm quan đến chức Thượng thư và từng được cử đi sứ nhà Minh năm 1488."

question = "nhà thiên văn nào đã đề xuất thuyết nhật tâm" # Answer: Nicolaus Co
context = "Nicolaus Copernicus đã phá vỡ quan niệm Trái đất nằm ở trung tâm của vũ trụ tồn tại suốt nhiều thế kỷ. Bằng những lập luận sắc bén trong thuyết nhật tâm, ông đề xuất rằng Trái đất và các hành tinh khác quay xung quanh Mặt trời."
# context = "Đối với Copernicus, lý thuyết nhật tâm của ông không hẳn là một bước ngoặt, bởi vì nó tạo ra nhiều vấn đề lớn cần phải giải quyết. Ví dụ, các vật thể nặng luôn được cho là rơi xuống mặt đất vì Trái đất là trung tâm của vũ trụ. Vậy tạo sao chúng lại rơi xuống đất trong một hệ thống lấy Mặt trời làm trung tâm?"
## NO ANSWER
# context = "Sau khi công trình cơ học thiên thể của Isaac Newton vào cuối thế kỷ 17 được công bố, sự chấp nhận thuyết nhật tâm lan truyền nhanh chóng ở các quốc gia ngoài Công giáo, và đến cuối thế kỷ 18, nó gần như được chấp nhận rộng rãi."
# context = "Hệ thống được ưa chuộng là hệ Ptolemy, trong đó Trái Đất nằm ở trung tâm vũ trụ và mọi thiên thể đều quay quanh nó. (Không nên lẫn lộn việc Cơ Đốc giáo ủng hộ thuyết địa tâm với ý tưởng về một Trái Đất phẳng, là cái chưa từng được Giáo hội ủng hộ.) Hệ Tycho đã sắp đặt ổn thỏa các vị trí của mô hình địa tâm, trong đó Mặt Trời quay quanh Trái Đất, trong khi các hành tinh quay quanh Mặt Trời giống như mô hình của Copernicus. Những nhà thiên văn học dòng Tên tại Roma ban đầu không đồng ý với hệ thống của Tycho; người nổi bật nhất là Clavius, ông đã bình luận rằng Tycho đã \"lẫn lộn mọi thứ trong thiên văn học, bởi vì ông muốn đặt Sao Hỏa thấp hơn Mặt Trời.\" (Fantoli, 2003, p. 109) Nhưng khi cuộc tranh cãi ngày càng phát triển và Giáo hội có quan điểm cứng rắn hơn về các ý tưởng của Copernicus sau năm 1616, dòng Tên quay sang ủng hộ việc giảng dạy ý tưởng của Tycho; sau năm 1633, việc sử dụng hệ thống này hầu như đã trở thành bắt buộc. Vì tội đã đề xuất thuyết nhật tâm, Galileo đã bị quản thúc tại gia trong nhiều năm."

# Tokenize input
inputs = tokenizer(question, context, return_tensors="pt", truncation=True)

with torch.no_grad():
    outputs = model(**inputs)

# Extract start/end logits
start_logits = outputs.start_logits
end_logits = outputs.end_logits

# Get the most probable start and end of answer
start_idx = torch.argmax(start_logits)
end_idx = torch.argmax(end_logits)

# Convert token indices to answer text
answer = tokenizer.convert_tokens_to_string(
    tokenizer.convert_ids_to_tokens(inputs["input_ids"][0][start_idx:end_idx + 1])
)

print("Answer:", answer)


Answer: <s>
